
# 🐼 EDA con Pandas — comparación con NumPy

En esta práctica repetimos el **mismo análisis exploratorio (EDA)** que hicimos con NumPy,  
pero ahora usando **Pandas**, para ver cómo cambia la sintaxis, el tiempo de desarrollo  
y las capacidades de análisis.

---

## 🧾 Preparación del dataset

In [17]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 1000

df = pd.DataFrame({
    "ID": np.arange(1, n+1),
    "Edad": np.random.randint(18, 65, size=n),
    "Salario": np.random.normal(2500, 800, size=n),
    "Experiencia": np.clip(np.random.randint(0, 40, size=n), 0, None),
    "Horas": np.random.normal(40, 5, size=n),
    "Nivel": np.random.randint(0, 5, size=n),
    "Satisfacción": np.random.normal(7, 2, size=n)
})

# Añadimos outliers y NaN
df.loc[np.random.randint(0, n, 10), "Salario"] *= 4
df.loc[np.random.randint(0, n, 15), "Satisfacción"] = np.nan

df.head(10)
df

,ID,Edad,Salario,Experiencia,Horas,Nivel,Satisfacción
0,1,56,3305.034247,9,36.191559,3,8.114609
1,2,46,2038.486504,4,41.549386,2,6.731710
2,3,32,3168.553690,35,39.852826,1,7.806938
3,4,60,1596.234516,5,51.391665,2,5.901280
4,5,25,2923.843342,25,42.733477,0,9.629560
...,...,...,...,...,...,...,...
995,996,22,2381.917931,10,34.356885,4,6.327323
996,997,40,2127.170799,12,42.339134,3,5.649254
997,998,27,1224.237546,10,50.406348,2,9.310601
998,999,61,2910.880085,30,41.229484,1,4.222246


---

## 🧠 Tarea 1: Descripción básica

In [18]:
print(df.shape)
print(df.columns)
df.head()

(1000, 7)
Index(['ID', 'Edad', 'Salario', 'Experiencia', 'Horas', 'Nivel',
       'Satisfacción'],
      dtype='object')


,ID,Edad,Salario,Experiencia,Horas,Nivel,Satisfacción
0,1,56,3305.034247,9,36.191559,3,8.114609
1,2,46,2038.486504,4,41.549386,2,6.731710
2,3,32,3168.553690,35,39.852826,1,7.806938
3,4,60,1596.234516,5,51.391665,2,5.901280
4,5,25,2923.843342,25,42.733477,0,9.629560


✅ **Más directo**: Pandas ya muestra etiquetas y formato de tabla.

---

## 📏 Tarea 2: Tipos y rangos

In [19]:
print(df.dtypes)
df[["Edad", "Salario"]].agg(["min", "max", "mean", "median", "std"])

ID                int64
Edad              int64
Salario         float64
Experiencia       int64
Horas           float64
Nivel             int64
Satisfacción    float64
dtype: object


,Edad,Salario
min,18.000000,182.995697
max,64.000000,14477.937397
mean,40.986000,2648.816048
median,42.000000,2567.549511
std,13.497852,1170.328194


✅ En una sola línea obtenemos estadísticas básicas.

---

## 💭 Tarea 3: Filtrado de datos

In [20]:
# 1. Salarios > 4000 €
print(df[df["Salario"] > 4000])

# 2. Más de 45h y >10 años experiencia
print(df[(df["Horas"] > 45) & (df["Experiencia"] > 10)][["ID", "Edad", "Horas", "Experiencia"]])

# 3. Porcentaje nivel ≥ 3
print((df["Nivel"] >= 3).mean() * 100)

      ID  Edad       Salario  Experiencia      Horas  Nivel  Satisfacción
23    24    38   4160.320639           27  36.250943      1      4.950015
25    26    29   8956.724697           34  38.897138      3     10.347803
45    46    31   4316.554286           14  41.209682      2      6.573056
67    68    35   4001.436650           20  45.515654      4      8.574062
76    77    25   4456.601584           33  40.460898      0      5.917749
84    85    64   4005.619597           33  40.315896      0      8.569563
116  117    29   4558.687843           13  40.666286      0     10.037437
120  121    41  10633.871234           25  28.796631      1      5.857251
181  182    30   9817.774004           37  34.195648      3      8.639776
217  218    49   4605.905652           20  29.969349      2      6.346753
224  225    41   4548.067631           16  32.777413      3     10.241952
225  226    22   9692.608321            4  47.151733      2      6.773839
236  237    20  11989.791879          

✅ Sintaxis más legible y declarativa.

---

## 🧹 Tarea 4: Limpieza — valores faltantes

In [21]:
print(df["Satisfacción"].isna().sum())
#df["Satisfacción"].fillna(df["Satisfacción"].mean(), inplace=True)
df["Satisfacción"]= df["Satisfacción"].fillna(df["Satisfacción"].mean())
print(df["Satisfacción"].isna().any())


15
False


✅ Un único método `fillna()` evita usar máscaras.

---

## ⚠️ Tarea 5: Detección de outliers

In [22]:
Q1 = df["Salario"].quantile(0.25)
Q3 = df["Salario"].quantile(0.75)
IQR = Q3 - Q1

filtro = (df["Salario"] < Q1 - 1.5*IQR) | (df["Salario"] > Q3 + 1.5*IQR)
df[filtro]

,ID,Edad,Salario,Experiencia,Horas,Nivel,Satisfacción
25,26,29,8956.724697,34,38.897138,3,10.347803
108,109,22,342.490686,3,38.206953,1,4.928178
120,121,41,10633.871234,25,28.796631,1,5.857251
130,131,54,379.224153,24,38.627800,0,7.193940
181,182,30,9817.774004,37,34.195648,3,8.639776
225,226,22,9692.608321,4,47.151733,2,6.773839
236,237,20,11989.791879,34,43.231487,3,5.119079
462,463,47,14477.937397,28,40.799824,3,8.558060
505,506,18,9876.772898,27,39.154244,0,5.821820
523,524,43,221.165903,4,43.576820,2,6.834279


✅ `quantile()` y condiciones combinadas simplifican la detección.

---

## 🧰 Tarea 6: Sustitución de outliers

In [23]:
p95 = df["Salario"].quantile(0.95)
df.loc[filtro, "Salario"] = p95

✅ Con `.loc[]` seleccionamos y sustituimos directamente.

---

## 📊 Tarea 7: Agrupaciones y resúmenes simples

In [24]:
# Media por nivel
print(df.groupby("Nivel")["Salario"].mean())
# Satisfacción media por grupo de edad
bins = [0, 30, 50, 100]
labels = ["<30", "30–50", ">50"]
df["GrupoEdad"] = pd.cut(df["Edad"], bins=bins, labels=labels)
print(df.groupby("GrupoEdad")["Satisfacción"].mean())

Nivel
0    2609.362141
1    2646.805404
2    2516.378435
3    2639.602694
4    2592.508181
Name: Salario, dtype: float64
GrupoEdad
<30      7.133667
30–50    6.862303
>50      7.091628
Name: Satisfacción, dtype: float64


/tmp/ipykernel_56191/3744473686.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("GrupoEdad")["Satisfacción"].mean())


✅ `groupby()` y `cut()` hacen triviales las operaciones por grupo.

---

## 🧩 Tarea 8: Normalización y escalado

In [25]:
df["Salario_norm"] = (df["Salario"] - df["Salario"].min()) / (df["Salario"].max() - df["Salario"].min())
df["Horas_z"] = (df["Horas"] - df["Horas"].mean()) / df["Horas"].std()
df[["Salario_norm", "Horas_z"]].describe()

,Salario_norm,Horas_z
count,1000.000000,1.000000e+03
mean,0.511417,2.877698e-16
std,0.194026,1.000000e+00
min,0.000000,-2.949402e+00
25%,0.377361,-6.733520e-01
50%,0.508072,2.185030e-02
75%,0.635730,6.573906e-01
max,1.000000,3.201389e+00


✅ Con Pandas, las operaciones vectorizadas siguen siendo de NumPy,
pero la sintaxis es más limpia y las columnas se guardan fácilmente.

---

## 🧮 Tarea 9: Correlaciones simples

In [27]:
print(df[["Salario", "Experiencia"]].corr())
print(df[["Edad", "Satisfacción"]].corr())

              Salario  Experiencia
Salario      1.000000     0.000423
Experiencia  0.000423     1.000000
                  Edad  Satisfacción
Edad          1.000000     -0.008994
Satisfacción -0.008994      1.000000


✅ `corr()` calcula la matriz de correlación directamente.

---

## 🧾 Tarea 10: Estadísticas generales

In [28]:
df.describe(include="all")

,ID,Edad,Salario,Experiencia,Horas,Nivel,Satisfacción,GrupoEdad,Salario_norm,Horas_z
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000,1000.000000,1.000000e+03
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30–50,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,438,NaN,NaN
mean,500.500000,40.986000,2600.061561,19.859000,40.132142,1.995000,7.002660,NaN,0.511417,2.877698e-16
std,288.819436,13.497852,796.560648,11.447102,5.051030,1.405329,1.955805,NaN,0.194026,1.000000e+00
min,1.000000,18.000000,500.475428,0.000000,25.234624,0.000000,0.449979,NaN,0.000000,-2.949402e+00
25%,250.750000,29.000000,2049.704208,10.000000,36.731021,1.000000,5.730191,NaN,0.377361,-6.733520e-01
50%,500.500000,42.000000,2586.330009,20.000000,40.242509,2.000000,6.909789,NaN,0.508072,2.185030e-02
75%,750.250000,52.000000,3110.421832,30.000000,43.452642,3.000000,8.318140,NaN,0.635730,6.573906e-01


✅ Una sola línea genera resumen completo de medias, desvíos,
mínimos, máximos y cuantiles — igual a lo que hicimos en 30 líneas con NumPy.

---

## ⚖️ Comparación final: NumPy vs Pandas

| Aspecto                       | NumPy                                     | Pandas                                         |
| ----------------------------- | ----------------------------------------- | ---------------------------------------------- |
| **Estructura**                | Arrays homogéneos (solo números)          | DataFrames con columnas heterogéneas           |
| **Acceso a datos**            | Por índice numérico                       | Por nombre de columna                          |
| **Limpieza (NaN, outliers)**  | Manual, usando máscaras y `np.isnan()`    | Métodos directos (`fillna`, `replace`, `clip`) |
| **Agrupaciones**              | Manual, por filtrado repetido             | `groupby`, `cut`, `pivot_table`                |
| **Resúmenes**                 | Funciones separadas (`mean`, `std`, etc.) | `describe()`, `agg()` integrados               |
| **Eficiencia**                | Muy rápido (array plano)                  | Un poco más lento, pero flexible               |
| **Lectura de datos externos** | No                                        | CSV, Excel, SQL, Parquet, JSON…                |

---

## 🧠 Reflexión: cuándo usar cada uno

* 🧮 **NumPy**: ideal para **cálculos numéricos puros**, operaciones de álgebra, vectores, tensores o machine learning a bajo nivel.
* 🐼 **Pandas**: ideal para **análisis de datos reales** con valores faltantes, mezcla de tipos, y manipulación de columnas o tablas grandes.

En la práctica, **Pandas usa NumPy por debajo** para los cálculos numéricos,
pero aporta una capa de **semántica tabular y limpieza** que simplifica enormemente el trabajo.

---

## ⚙️ Pandas y sus motores internos

Desde **Pandas 2.0**, el DataFrame puede usar distintos **motores de ejecución**:

| Motor         | Descripción                                                 | Uso principal                                                                             |
| ------------- | ----------------------------------------------------------- | ----------------------------------------------------------------------------------------- |
| 🧩 **NumPy**  | Motor tradicional basado en arrays NumPy (`dtype` clásico). | Por defecto en la mayoría de entornos.                                                    |
| ⚡ **PyArrow** | Motor alternativo basado en Apache Arrow (`dtype="arrow"`). | Mayor eficiencia, tipos nulos nativos, interoperabilidad con Parquet, DuckDB, Spark, etc. |

Puedes comprobar o cambiar el motor por defecto:

In [ ]:
import pandas as pd
pd.options.mode.dtype_backend
# 'numpy' o 'pyarrow'

OptionError: You can only set the value of existing options

Y cambiarlo así:

In [30]:
pd.set_option("mode.dtype_backend", "pyarrow")

OptionError: No such keys(s): 'mode.dtype_backend'

💡 **Ventajas de PyArrow**:

* Maneja nulos reales, no solo `NaN`
* Más rápido en operaciones columnares
* Reduce consumo de memoria
* Integración con otros frameworks (Arrow, Polars, DuckDB, etc.)

---

## 🧾 En resumen

| Característica | NumPy                                  | Pandas                       |
| -------------- | -------------------------------------- | ---------------------------- |
| Nivel          | Bajo nivel (arrays)                    | Alto nivel (tablas)          |
| Ideal para     | Cálculos numéricos, ML, algebra lineal | Limpieza, análisis, EDA, ETL |
| Falta de datos | Manejo limitado                        | Soporte completo             |
| Tipos de datos | Homogéneos                             | Heterogéneos                 |
| Motores        | —                                      | NumPy o PyArrow              |

---

## 🚀 Próximo paso

➡️ **Repite el EDA con Pandas y compara tiempos, líneas y legibilidad.**
➡️ Después exploraremos **visualización (Matplotlib / Seaborn)** para completar el análisis exploratorio.

```

---